## Getting Started

To reproduce results:
1. Upload data files located in `evergreen_analyzers/Notebooks/week3_artifacts` to jupyterhub root
2. Update `ITERATION_NUMBER` in Step 2 to track different runs and any other path and/or file name
3. Run all cells in order from Step 1 to Step 7
4. Results will be saved to `ROOT / Results`

## Note: code was borrowed from example provided and modified to meet our needs

# Step 1: Imports Results
What this does: imports the tools needed for file paths, table operations, parsing model
output, and calling the model endpoint.

Why it matters: clean imports at the top make each later step focused on logic, not setup.

Principle: set up your toolbox first so experiments are repeatable and easy to debug.

Note: Code from example provided.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import openai
from openai import OpenAI

# Find the data file
matches = list(Path('.').rglob('data_cleaned_split.csv'))
print("Found at:", matches)


Found at: [PosixPath('data_cleaned_split.csv')]


# Step 2: Paths and settings
What this does: sets all configurable values in one place — input files, column names,
model settings, and output file locations.

Why it matters: centralizing settings means you only need to change one place when
file locations or model endpoints change.

Principle: separate configuration from logic. Hard-coded paths buried in code are
difficult to find, update, and audit.

Note: must upload data files in jupyterHub at same leavel as notebook
Note: example code modified

In [2]:
ROOT = Path('.').resolve()

WEEK3_TEST_CSV = ROOT / 'data_cleaned_split.csv'

SITE_ID_COL = "monitoring_location_id"
SITES_TO_PREDICT = ["USGS-12462500", "USGS-12422500"]
TARGET_COLUMN = 'flow_rate'
START_DATE = '2022-08-13'

MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

# Create Results and Prompts directory if they do not exist
OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR = ROOT / 'Prompts'
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

def get_unique_path(base_path):
    path = Path(base_path)
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    parent = path.parent
    i = 1
    while True:
        new_path = parent / f"{stem}_{i}{suffix}"
        if not new_path.exists():
            return new_path
        i += 1

RESULTS_PATH = get_unique_path(OUTPUT_DIR / f'llm_results_{ITERATION_NUMBER}.csv')


# Step 3: Load Week 3 files
What this does: loads Week 3 test data and Week 3 ML predictions, validates expected
columns, then merges them on monitoring_location_id.

Why it matters: Week 4 comparison is only fair if both ML and LLM use the same
held-out test examples.

Principle: benchmark validity depends on data integrity. Always validate schema and
join keys before scoring.

In [3]:
df = pd.read_csv(WEEK3_TEST_CSV, parse_dates=['time'])

# Restrict to the relevant sites
df = df[df[SITE_ID_COL].apply(lambda x: x in SITES_TO_PREDICT)]

# Restrict to recent data for context window reasons
df = df[df['time'] >= START_DATE]

# Check for missing columns
if TARGET_COLUMN not in df.columns:
    raise ValueError(f'Missing in test_split.csv: {target_column}')

# Restrict to relevant columns
df = df[['time'] + [TARGET_COLUMN] + [SITE_ID_COL] + ['split']]    
    
print(f'Loaded rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print(df['split'].value_counts())
display(df.head())

Loaded rows: 2652
Columns: ['time', 'flow_rate', 'monitoring_location_id', 'split']
split
test    1920
val      732
Name: count, dtype: int64


,time,flow_rate,monitoring_location_id,split
8260,2022-08-13 00:00:00+00:00,1600.0,USGS-12422500,val
8261,2022-08-14 00:00:00+00:00,1620.0,USGS-12422500,val
8262,2022-08-15 00:00:00+00:00,1400.0,USGS-12422500,val
8263,2022-08-16 00:00:00+00:00,1140.0,USGS-12422500,val
8264,2022-08-17 00:00:00+00:00,1020.0,USGS-12422500,val


# Step 4: Prompt + parser helpers
What this does: defines flow classification bins, a prompt template that sends flow rate
and rolling average to the model, and a parser that turns model text into structured
fields — prediction, confidence, and parse status.

Why it matters: LLM responses are free-form by default but evaluation requires
deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear
output contracts plus defensive checks.

In [4]:
def make_prompt(df, site_id):
    df = df[df[SITE_ID_COL] == site_id]
    train_val_df = df[df['split'].apply(lambda x: x in ['train', 'val'])]
    time_min = train_val_df['time'].min()
    time_max = train_val_df['time'].max()
    next_n_days = (df['split'] == 'test').sum()
    return (
        f"The following time series gives daily flow rate data for a single monitoring "
        f"site in average cubic feet per second, from {time_min} to {time_max}, a total "
        f"of {train_val_df.shape[0]} days. "
        f"Predict the next {next_n_days} days of flow rate data, paying attention to "
        f"seasonality and other important factors in the given data.\n"
        f"Your output should simply be a comma separated list of floating point numbers. "
        f"No extra text. Do not include brackets or anything other than numbers and commas. \n"
        f"Here is the previous data: {train_val_df[TARGET_COLUMN].tolist()}."
    )

def parse_response(raw_text):
    output = {
        'llm_prediction': None, 
        'parse_ok': False, 
        'parse_error': None,
    }

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output
    
    try:
        result = raw_text.replace(' ', '').split(',')
        result = [float(a) for a in result]
        output['llm_prediction'] = result
        output['parse_ok'] = True
        return output
    except Exception as E:
        output['parse_error'] = E
        return output
    
    
print("Example prompt:")
print(make_prompt(df, SITES_TO_PREDICT[0]))

Example prompt:
The following time series gives daily flow rate data for a single monitoring site in average cubic feet per second, from 2022-08-13 00:00:00+00:00 to 2023-08-13 00:00:00+00:00, a total of 366 days. Predict the next 960 days of flow rate data, paying attention to seasonality and other important factors in the given data.
Your output should simply be a comma separated list of floating point numbers. No extra text. Do not include brackets or anything other than numbers and commas. 
Here is the previous data: [1320.0, 1260.0, 1170.0, 1110.0, 1070.0, 1050.0, 1040.0, 1030.0, 1020.0, 996.0, 941.0, 903.0, 885.0, 850.0, 839.0, 815.0, 761.0, 718.0, 685.0, 674.0, 666.0, 655.0, 663.0, 625.0, 601.0, 581.0, 556.0, 536.0, 517.0, 504.0, 501.0, 517.0, 537.0, 548.0, 538.0, 523.0, 512.0, 498.0, 479.0, 460.0, 448.0, 431.0, 419.0, 410.0, 409.0, 407.0, 405.0, 409.0, 407.0, 395.0, 394.0, 393.0, 391.0, 392.0, 392.0, 422.0, 468.0, 464.0, 460.0, 461.0, 444.0, 437.0, 434.0, 432.0, 425.0, 419.0, 4

# Step 5: Single-row smoke test
What this does: tests one example row on each model endpoint before running the full
benchmark loop.

Why it matters: this catches endpoint and port issues early and confirms all three
models are reachable and returning parseable output.

Principle: validate connectivity and output format for each model before large-scale
evaluation. Fail fast on one row, not after thousands of API calls.

In [5]:
SYSTEM_PROMPT = 'You are a careful classifier that follows output format instructions.'

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(make_prompt(df, SITES_TO_PREDICT[0]), endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"Model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print(f"Raw response: {raw[:100]} ... ")
    if not parsed['parse_error']:
        print(f"Parsed: {parsed['llm_prediction'][:50]}...")
    print()


Model: Phi-3.5-mini-instruct (port 8000)
Raw response:  1320.0,1260.0,1170.0,1110.0,1070.0,1050.0,1040.0,1030.0,1020.0,996.0,941.0,903.0,885.0,850.0,839.0, ... 
Parsed: [1320.0, 1260.0, 1170.0, 1110.0, 1070.0, 1050.0, 1040.0, 1030.0, 1020.0, 996.0, 941.0, 903.0, 885.0, 850.0, 839.0, 815.0, 761.0, 718.0, 685.0, 674.0, 666.0, 655.0, 663.0, 625.0, 601.0, 581.0, 556.0, 536.0, 517.0, 504.0, 501.0, 517.0, 537.0, 548.0, 538.0, 523.0, 512.0, 498.0, 479.0, 460.0, 448.0, 431.0, 419.0, 410.0, 409.0, 407.0, 405.0, 409.0, 407.0, 395.0]...

Model: Phi-mini-MoE-instruct (port 8001)
Raw response:  1320.0,1260.0,1170.0,1110.0,1070.0,1050.0,1040.0,1030.0,1010.0,996.0,941.0,903.0,885.0,850.0,839.0, ... 
Parsed: [1320.0, 1260.0, 1170.0, 1110.0, 1070.0, 1050.0, 1040.0, 1030.0, 1010.0, 996.0, 941.0, 903.0, 885.0, 850.0, 839.0, 817.0, 761.0, 718.0, 685.0, 674.0, 655.0, 656.0, 638.0, 625.0, 601.0, 580.0, 561.0, 550.0, 540.0, 535.0, 525.0, 515.0, 505.0, 501.0, 500.0, 500.0, 500.0, 500.0, 500.0, 500.0, 500.0, 5

# Step 5b Benchmark loop 
What this does: runs every model endpoint against a sampled subset of the test data,
collects raw and parsed responses, and saves results to CSV.

Why it matters: a single smoke test row is not enough to evaluate model performance.
Running across many examples reveals how models behave across the full range of flow values.

Principle: sample before you scale. Start with a small subset to confirm the loop runs
cleanly end-to-end before committing to the full 479k rows.


In [6]:
print(df['split'].value_counts())
print(f"\nTotal rows: {len(df)}")
print(f"\nTest rows only: {len(df[df['split'] == 'test'])}")

split
test    1920
val      732
Name: count, dtype: int64

Total rows: 2652

Test rows only: 1920


In [7]:
import time
from tqdm import tqdm

TEMPERATURE = 0.0
SEED = 0

all_results = []

for endpoint_cfg in MODEL_ENDPOINTS:
    for site_id in SITES_TO_PREDICT:
        
        print(f"\nRunning model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
        
        try:
            raw = query_llm(make_prompt(df, site_id), endpoint_cfg, temperature=0.0, seed=0)
            parsed = parse_response(raw)
        except Exception as E:
            print(f"Failed with {endpoint_cfg['model_name']} on site {site_id}")
            continue
        
        all_results.append({
            'model': endpoint_cfg['label'],
            'site': site_id,
            **parsed,
        })

    print(f"Done with {endpoint_cfg['model_name']}")



Running model: Phi-3.5-mini-instruct (port 8000)

Running model: Phi-3.5-mini-instruct (port 8000)
Done with Phi-3.5-mini-instruct

Running model: Phi-mini-MoE-instruct (port 8001)

Running model: Phi-mini-MoE-instruct (port 8001)
Done with Phi-mini-MoE-instruct

Running model: gemma-3-12b-it (port 8002)

Running model: gemma-3-12b-it (port 8002)
Done with gemma-3-12b-it


In [8]:
print(f'{sum([r["parse_ok"] for r in all_results])} out of {len(all_results)} LLM result parses succeeded')

6 out of 6 LLM result parses succeeded


In [9]:
# Print error cases if any
for r in all_results:
    if not r['parse_ok']:
        print(r['model'])
        print(r['parse_error'])

In [10]:
print('Lengths of parsed predictions:')
print('\n'.join([str(len(r['llm_prediction'])) for r in all_results if r['parse_ok']]))

Lengths of parsed predictions:
186
147
193
138
366
366


In [ ]:
# Place results in dataframe and save
results_df = df.copy()

for endpoint_cfg in MODEL_ENDPOINTS:
    results_df[endpoint_cfg["label"]] = np.nan

for result in all_results:
    if not result['parse_ok']:
        continue
    preds = result['llm_prediction']
    idx = (results_df.loc[(results_df["monitoring_location_id"] == result['site']) & (results_df["split"] == "test")]
             .sort_values("time")
             .index)
    n_entries = min(len(preds), len(idx))
    results_df.loc[idx[:n_entries], result['model']] = preds[:n_entries]

# Save one CSV per model
for model_label in [endpoint_cfg['label'] for endpoint_cfg in MODEL_ENDPOINTS]:
    model_df = results_df[["time", "flow_rate", "monitoring_location_id", "split", model_label]].copy()
    model_path = OUTPUT_DIR / f'llm_results_{model_label}_iter{ITERATION_NUMBER}.csv'
    model_df.to_csv(model_path, index=False)
    print(f"Saved: {model_path} ({len(model_df)} rows)")

results_df.to_csv(RESULTS_PATH, index=False)

test_cutoff = results_df[results_df['split'] == 'test']['time'].min()
display((results_df[results_df['time'] >= test_cutoff].head()))

,time,flow_rate,monitoring_location_id,split,phi-3.5-mini-instruct,phi-mini-moe-instruct,gemma-3-12b-it
8626,2023-08-14 00:00:00+00:00,807.0,USGS-12422500,test,1600.0,1600.0,1600.0
8627,2023-08-15 00:00:00+00:00,780.0,USGS-12422500,test,1620.0,1620.0,1620.0
8628,2023-08-16 00:00:00+00:00,772.0,USGS-12422500,test,1400.0,1400.0,1400.0
8629,2023-08-17 00:00:00+00:00,767.0,USGS-12422500,test,1140.0,1140.0,1140.0
8630,2023-08-18 00:00:00+00:00,753.0,USGS-12422500,test,1020.0,1099.0,1020.0


# Step 6  
What this does: scores model outputs according to metrics from previous week (root mean squared error) and summarizes missing values. 

Why it matters: Need to compare apples-to-apples for benchmarking LLM vs ML models, and maintain awareness of limitations in LLM outputs.

In [37]:
model_labels = [endpoint_cfg['label'] for endpoint_cfg in MODEL_ENDPOINTS]
model_performance = {
    l: {
        site_id: {} for site_id in SITES_TO_PREDICT
    } for l in model_labels
}
for model_label in model_labels:
    print(f'Evaluating {model_label}')
    for site_id in SITES_TO_PREDICT:
        print(f'   Site {site_id}:')
        test_data = results_df[(results_df['split'] == 'test')&(results_df['monitoring_location_id'] ==site_id)].sort_values('time')['flow_rate'].values
        model_preds = results_df[(results_df['split'] == 'test')&(results_df['monitoring_location_id'] ==site_id)].sort_values('time')[model_label].values
        model_performance[model_label][site_id]['missing_count'] = int(np.isnan(model_preds).sum())
        model_performance[model_label][site_id]['missing_pct'] = float(np.isnan(model_preds).sum()/test_data.size)
        model_performance[model_label][site_id]['rmse'] = float(np.sqrt(np.nanmean(np.square((model_preds-test_data)))))
        print(f'      RMSE: {model_performance[model_label][site_id]["rmse"]}')
        print(f'      Missing values: {model_performance[model_label][site_id]["missing_count"]} '
               f'({100*model_performance[model_label][site_id]["missing_pct"]:.2f}%)')
    print('')

Evaluating phi-3.5-mini-instruct
   Site USGS-12462500:
      RMSE: 1388.092103634353
      Missing values: 774 (80.62%)
   Site USGS-12422500:
      RMSE: 812.0623405243954
      Missing values: 813 (84.69%)

Evaluating phi-mini-moe-instruct
   Site USGS-12462500:
      RMSE: 1555.2547148230333
      Missing values: 767 (79.90%)
   Site USGS-12422500:
      RMSE: 1990.8745893287157
      Missing values: 822 (85.62%)

Evaluating gemma-3-12b-it
   Site USGS-12462500:
      RMSE: 2791.154884660458
      Missing values: 594 (61.88%)
   Site USGS-12422500:
      RMSE: 4386.088071821004
      Missing values: 594 (61.88%)

